In [25]:
import pandas as pd

df = pd.read_csv(r"C:\Users\louis\Downloads\R&D PROJECTS\data\V2-nigerian-financial-transactions-and-fraud-detection-dataset-for-model-training.csv")

In [ ]:
df.info()

In [5]:
df.head()

,transaction_id,timestamp,sender_account,receiver_account,transaction_type,merchant_category,location,device_used,is_fraud,fraud_type,...,txn_count_last_1h,txn_count_last_24h,total_amount_last_1h,time_since_last,avg_gap_between_txns,merchant_fraud_rate,channel_risk_score,persona_fraud_risk,location_fraud_risk,ip_geo_region
0,T2162315,2023-01-24 09:54:06.198396,1000018177,8385560081,deposit,Local Market Purchase,Aba,mobile,False,NaN,...,1,1,654135.08,0.000000,0.000000,0.1,0.8,0.5,0.1,South East
1,T1764581,2023-02-22 16:16:19.271951,1000018177,5643014197,payment,SPAR Purchase,Onitsha,mobile,False,NaN,...,2,2,687449.11,42142.217893,21071.108946,0.1,0.3,0.5,0.1,South East
2,T3305551,2023-05-04 16:01:42.312142,1000018177,7722691989,withdrawal,Other Transaction,Onitsha,web,False,NaN,...,3,3,719985.77,102225.384003,48122.533965,0.1,0.8,0.5,0.0,South East
3,T174955,2023-05-07 13:15:03.037215,1000018177,4987435115,payment,Ikeja Electric Bill,Benin City,atm,False,NaN,...,4,4,733430.88,4153.345418,37130.236828,0.1,0.3,0.5,0.1,South South
4,T3695059,2023-06-08 11:37:39.155188,1000018177,7939643449,withdrawal,Arik Air Flight,Aba,web,False,NaN,...,5,5,858543.44,45982.601966,38900.709856,0.1,0.6,0.5,0.0,South East


In [36]:
df.shape

(5000000, 44)

In [37]:
df.isnull().sum().sort_values(ascending=False)


fraud_type                     4820447
transaction_id                       0
user_top_category                    0
device_seen_count                    0
is_device_shared                     0
ip_seen_count                        0
is_ip_shared                         0
user_txn_count_total                 0
user_avg_txn_amt                     0
user_std_txn_amt                     0
user_txn_frequency_24h               0
txn_count_last_1h                    0
is_salary_week                       0
txn_count_last_24h                   0
total_amount_last_1h                 0
time_since_last                      0
avg_gap_between_txns                 0
merchant_fraud_rate                  0
channel_risk_score                   0
persona_fraud_risk                   0
location_fraud_risk                  0
is_night_txn                         0
is_weekend                           0
sender_account                       0
velocity_score                       0
receiver_account         

In [38]:
#drop identifier columns - they don't describe behaviour

df = df.drop(columns=[
    'transaction_id',
    'sender_account',
    'receiver_account',
    'ip_address',
    'device_hash',
    'fraud_type'
])

In [39]:
df.columns

Index(['transaction_type', 'merchant_category', 'location', 'device_used',
       'is_fraud', 'time_since_last_transaction', 'spending_deviation_score',
       'velocity_score', 'geo_anomaly_score', 'payment_channel', 'amount_ngn',
       'bvn_linked', 'new_device_transaction', 'sender_persona',
       'geospatial_velocity_anomaly', 'txn_hour', 'is_weekend',
       'is_salary_week', 'is_night_txn', 'device_seen_count',
       'is_device_shared', 'ip_seen_count', 'is_ip_shared',
       'user_txn_count_total', 'user_avg_txn_amt', 'user_std_txn_amt',
       'user_txn_frequency_24h', 'user_top_category', 'txn_count_last_1h',
       'txn_count_last_24h', 'total_amount_last_1h', 'time_since_last',
       'avg_gap_between_txns', 'merchant_fraud_rate', 'channel_risk_score',
       'persona_fraud_risk', 'location_fraud_risk', 'ip_geo_region'],
      dtype='object')

In [40]:
# handle missing values - models cannot process missing values
# replacing the time since last transaction missing values with median

df['time_since_last_transaction'].fillna(
    df['time_since_last_transaction'].median(),
    inplace=True
)

C:\Users\louis\AppData\Local\Temp\ipykernel_15536\3318292799.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['time_since_last_transaction'].fillna(


In [41]:
# convert boolean features - True/False -> 1/0

bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

## LIGHTGBM 

In [42]:
# define our categorical columns
cat_cols = [
    'transaction_type', 'merchant_category', 'location',
    'device_used', 'payment_channel', 'sender_persona',
    'user_top_category', 'ip_geo_region'
]


In [43]:
# convert categorical columns to category dtype
for col in cat_cols:
    if col in df.columns:
        df[col] = df[col].astype('category')


In [44]:
df.isnull().sum()

transaction_type               0
merchant_category              0
location                       0
device_used                    0
is_fraud                       0
time_since_last_transaction    0
spending_deviation_score       0
velocity_score                 0
geo_anomaly_score              0
payment_channel                0
amount_ngn                     0
bvn_linked                     0
new_device_transaction         0
sender_persona                 0
geospatial_velocity_anomaly    0
txn_hour                       0
is_weekend                     0
is_salary_week                 0
is_night_txn                   0
device_seen_count              0
is_device_shared               0
ip_seen_count                  0
is_ip_shared                   0
user_txn_count_total           0
user_avg_txn_amt               0
user_std_txn_amt               0
user_txn_frequency_24h         0
user_top_category              0
txn_count_last_1h              0
txn_count_last_24h             0
total_amou

In [45]:
# train / validation / test SPLIT
# 70% train, 15% validation, 15% test 
# MUST be split stratified because fraud is rare
from sklearn.model_selection import train_test_split

X = df.drop('is_fraud', axis=1)
y = df['is_fraud']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)


In [16]:
pip install lightgbm jupyter pandas numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [46]:
df = df.drop(columns=['timestamp'])

KeyError: "['timestamp'] not found in axis"

In [47]:
# Train LIGHTGBM


from lightgbm import LGBMClassifier, early_stopping, log_evaluation

model = LGBMClassifier(
    objective='binary',
    learning_rate=0.05,
    num_leaves=64,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=5,
    n_estimators=500,
    scale_pos_weight=(len(y_train) - y_train.sum()) / y_train.sum()
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='auc',
    callbacks=[
        early_stopping(stopping_rounds=50),
        log_evaluation(period=50)
    ]
)


[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Info] Number of positive: 125687, number of negative: 3374313
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.351393 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3057
[LightGBM] [Info] Number of data

,boosting_type,'gbdt'
,num_leaves,64
,max_depth,-1
,learning_rate,0.05
,n_estimators,500
,subsample_for_bin,200000
,objective,'binary'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [48]:
from sklearn.metrics import roc_auc_score, average_precision_score

y_pred = model.predict_proba(X_test)[:, 1]

print("AUC:", roc_auc_score(y_test, y_pred))
print("AUC-PR:", average_precision_score(y_test, y_pred))


[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
AUC: 0.5929513505782891
AUC-PR: 0.04382550458811774


## AUC: 0.5929
## AUC-PR: 0.0438
WEAK model - but this is the first baseline.

AUC ≈ 0.59
The model is only slightly better than random guessing (0.50).
For fraud detection, we want 0.75+ at minimum.

AUC‑PR ≈ 0.044
This is low, but fraud datasets are extremely imbalanced.
If fraud is 0.5% of your data, a random model gets ~0.005.
0.044 is actually 8× better than random, but still not good enough.

## NEXT STEPS - TUNING / REFINING
- Step 1 — Tune LightGBM properly 
- Step 2 — Add stronger behavioural features
- Step 3 — Add merchant/device/IP history features
- Step 4 — Use monotonic constraints for stability
- Step 5 — Use SHAP to understand what’s working 

In [49]:
# More trees + early stopping → model can grow until it stops improving.
# More leaves → can capture complex fraud patterns.
# min_data_in_leaf + L1/L2 → prevents overfitting.
# scale_pos_weight → tells the model fraud is rare but important.
# feature/bagging fractions → add randomness and robustness.



from lightgbm import LGBMClassifier, early_stopping, log_evaluation

pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()

model = LGBMClassifier(
    objective='binary',
    boosting_type='gbdt',
    learning_rate=0.03,
    n_estimators=3000,          # many trees + early stopping
    num_leaves=128,             # more complex trees
    max_depth=-1,               # let it grow, controlled by leaves
    min_data_in_leaf=200,       # regularization
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=5,
    lambda_l1=1.0,
    lambda_l2=2.0,
    scale_pos_weight=pos_weight,
    n_jobs=-1
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='auc',
    callbacks=[
        early_stopping(stopping_rounds=200),
        log_evaluation(period=100)
    ]
)


[LightGBM] [Warning] min_data_in_leaf is set=200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=200
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] lambda_l1 is set=1.0, reg_alpha=0.0 will be ignored. Current value: lambda_l1=1.0
[LightGBM] [Warning] lambda_l2 is set=2.0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=2.0
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] min_data_in_leaf is set=200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=200
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] lambda_l1 is set=1.0, reg_alpha=0.0 will be ignored

,boosting_type,'gbdt'
,num_leaves,128
,max_depth,-1
,learning_rate,0.03
,n_estimators,3000
,subsample_for_bin,200000
,objective,'binary'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [50]:
# Re-evaluate on test set

from sklearn.metrics import roc_auc_score, average_precision_score

y_pred = model.predict_proba(X_test)[:, 1]

print("AUC:", roc_auc_score(y_test, y_pred))
print("AUC-PR:", average_precision_score(y_test, y_pred))


[LightGBM] [Warning] min_data_in_leaf is set=200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=200
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] lambda_l1 is set=1.0, reg_alpha=0.0 will be ignored. Current value: lambda_l1=1.0
[LightGBM] [Warning] lambda_l2 is set=2.0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=2.0
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
AUC: 0.5931356441752516
AUC-PR: 0.04385327412907335


The AUR or AUC - PR score didn't improve meaning the issue is not with LightGBM tuning, it's the features we need to improve.
MOVING ONTO FEATURE ENGINEERING

In [51]:
df.columns


Index(['transaction_type', 'merchant_category', 'location', 'device_used',
       'is_fraud', 'time_since_last_transaction', 'spending_deviation_score',
       'velocity_score', 'geo_anomaly_score', 'payment_channel', 'amount_ngn',
       'bvn_linked', 'new_device_transaction', 'sender_persona',
       'geospatial_velocity_anomaly', 'txn_hour', 'is_weekend',
       'is_salary_week', 'is_night_txn', 'device_seen_count',
       'is_device_shared', 'ip_seen_count', 'is_ip_shared',
       'user_txn_count_total', 'user_avg_txn_amt', 'user_std_txn_amt',
       'user_txn_frequency_24h', 'user_top_category', 'txn_count_last_1h',
       'txn_count_last_24h', 'total_amount_last_1h', 'time_since_last',
       'avg_gap_between_txns', 'merchant_fraud_rate', 'channel_risk_score',
       'persona_fraud_risk', 'location_fraud_risk', 'ip_geo_region'],
      dtype='object')

## ADD STRONG INTERACTION FEATURES

In [52]:
# --- INTERACTION FEATURES ---

# 1. Amount relative to user's normal behaviour
df['amt_to_avg_ratio'] = df['amount_ngn'] / (df['user_avg_txn_amt'] + 1)
df['amt_to_std_ratio'] = df['amount_ngn'] / (df['user_std_txn_amt'] + 1)

# 2. Device/IP anomaly strength
df['device_ip_ratio'] = df['device_seen_count'] / (df['ip_seen_count'] + 1)

# 3. Velocity × anomaly interaction
df['velocity_geo_interaction'] = df['velocity_score'] * df['geo_anomaly_score']

# 4. Time-based spending patterns
df['night_amount'] = df['amount_ngn'] * df['is_night_txn']
df['weekend_amount'] = df['amount_ngn'] * df['is_weekend']

# 5. Risk-weighted spending
df['merchant_risk_amount'] = df['merchant_fraud_rate'] * df['amount_ngn']
df['channel_risk_amount'] = df['channel_risk_score'] * df['amount_ngn']
df['persona_risk_amount'] = df['persona_fraud_risk'] * df['amount_ngn']
df['location_risk_amount'] = df['location_fraud_risk'] * df['amount_ngn']


In [53]:
# recreate x and y

X = df.drop('is_fraud', axis=1)
y = df['is_fraud']


In [54]:
# train / validate / test again

from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)


In [55]:
# train lightgbm 

from lightgbm import LGBMClassifier, early_stopping, log_evaluation

pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()

model = LGBMClassifier(
    objective='binary',
    boosting_type='gbdt',
    learning_rate=0.03,
    n_estimators=3000,
    num_leaves=128,
    max_depth=-1,
    min_data_in_leaf=200,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=5,
    lambda_l1=1.0,
    lambda_l2=2.0,
    scale_pos_weight=pos_weight,
    n_jobs=-1
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='auc',
    callbacks=[
        early_stopping(stopping_rounds=200),
        log_evaluation(period=100)
    ]
)


[LightGBM] [Warning] min_data_in_leaf is set=200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=200
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] lambda_l1 is set=1.0, reg_alpha=0.0 will be ignored. Current value: lambda_l1=1.0
[LightGBM] [Warning] lambda_l2 is set=2.0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=2.0
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] min_data_in_leaf is set=200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=200
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] lambda_l1 is set=1.0, reg_alpha=0.0 will be ignored

,boosting_type,'gbdt'
,num_leaves,128
,max_depth,-1
,learning_rate,0.03
,n_estimators,3000
,subsample_for_bin,200000
,objective,'binary'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [56]:
from sklearn.metrics import roc_auc_score, average_precision_score

y_pred = model.predict_proba(X_test)[:, 1]

print("AUC:", roc_auc_score(y_test, y_pred))
print("AUC-PR:", average_precision_score(y_test, y_pred))


[LightGBM] [Warning] min_data_in_leaf is set=200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=200
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] lambda_l1 is set=1.0, reg_alpha=0.0 will be ignored. Current value: lambda_l1=1.0
[LightGBM] [Warning] lambda_l2 is set=2.0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=2.0
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
AUC: 0.5942895758037389
AUC-PR: 0.044031749493091694


Our dataset is already pre-engineered
It contains:
- time_since_last_transaction
- txn_count_last_1h
- txn_count_last_24h
- user_avg_txn_amt
- user_std_txn_amt
- merchant_fraud_rate
- channel_risk_score
- persona_fraud_risk
- location_fraud_risk
- device_seen_count
- ip_seen_count
These are already the strongest features normally built from raw logs.

## REMOVE HARMFUL FEATURES
- highly correlated
- noisy
- derived from the same base signals 

In [57]:
# remove harmful features 
cols_to_drop = [
    'time_since_last',            # duplicate of time_since_last_transaction
    'avg_gap_between_txns',       # redundant with time_since_last_transaction
    'user_txn_count_total',       # often just correlates with user_avg_txn_amt
    'user_txn_frequency_24h',     # redundant with txn_count_last_24h
    'txn_count_last_1h',          # redundant with velocity_score
    'txn_count_last_24h',         # redundant with velocity_score
    'total_amount_last_1h',       # redundant with amount_ngn + velocity
    'spending_deviation_score',   # often noisy
    'geo_anomaly_score',          # often noisy unless raw geo data exists
    'geospatial_velocity_anomaly' # redundant with velocity_score
]

df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])



In [58]:
# lightgbm again
from lightgbm import LGBMClassifier, early_stopping, log_evaluation

# Handle class imbalance
pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()

model = LGBMClassifier(
    objective='binary',
    boosting_type='gbdt',
    learning_rate=0.03,
    n_estimators=3000,          # many trees + early stopping
    num_leaves=128,             # more complex trees
    max_depth=-1,               # unlimited depth, controlled by leaves
    min_data_in_leaf=200,       # regularization
    feature_fraction=0.8,       # column sampling
    bagging_fraction=0.8,       # row sampling
    bagging_freq=5,
    lambda_l1=1.0,              # L1 regularization
    lambda_l2=2.0,              # L2 regularization
    scale_pos_weight=pos_weight,
    n_jobs=-1
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='auc',
    callbacks=[
        early_stopping(stopping_rounds=200),
        log_evaluation(period=100)
    ]
)


[LightGBM] [Warning] min_data_in_leaf is set=200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=200
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] lambda_l1 is set=1.0, reg_alpha=0.0 will be ignored. Current value: lambda_l1=1.0
[LightGBM] [Warning] lambda_l2 is set=2.0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=2.0
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] min_data_in_leaf is set=200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=200
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] lambda_l1 is set=1.0, reg_alpha=0.0 will be ignored

,boosting_type,'gbdt'
,num_leaves,128
,max_depth,-1
,learning_rate,0.03
,n_estimators,3000
,subsample_for_bin,200000
,objective,'binary'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [59]:
# reevaluate
from sklearn.metrics import roc_auc_score, average_precision_score

y_pred = model.predict_proba(X_test)[:, 1]

print("AUC:", roc_auc_score(y_test, y_pred))
print("AUC-PR:", average_precision_score(y_test, y_pred))


[LightGBM] [Warning] min_data_in_leaf is set=200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=200
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] lambda_l1 is set=1.0, reg_alpha=0.0 will be ignored. Current value: lambda_l1=1.0
[LightGBM] [Warning] lambda_l2 is set=2.0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=2.0
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
AUC: 0.5942895758037389
AUC-PR: 0.044031749493091694


Model is stuck at ~0.59 AUC because:
- Don’t have raw timestamps
- Don’t have user/device/IP/merchant identifiers
- Can’t build rolling windows or sequence features
- The dataset is already pre‑engineered and flattened
- LightGBM has extracted all available signal
This is a data limitation, not a modelling issue.

In [61]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_test)
y_pred_binary = (y_pred > 0.5).astype(int)   # threshold for fraud

print(classification_report(y_test, y_pred_binary))


[LightGBM] [Warning] min_data_in_leaf is set=200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=200
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] lambda_l1 is set=1.0, reg_alpha=0.0 will be ignored. Current value: lambda_l1=1.0
[LightGBM] [Warning] lambda_l2 is set=2.0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=2.0
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
              precision    recall  f1-score   support

           0       0.96      1.00      0.98    723067
           1       0.00      0.00      0.00     26933

    accuracy                           0.96    750000
   macro avg       0.48      0.50      0.49    750000
weighted avg       0.93      0.96      0.95    7

C:\Users\louis\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\louis\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\louis\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


MODEL IS PREDICTING 0 for everything

In [100]:
y_pred_binary = (y_pred > 0.05).astype(int)
print(classification_report(y_test, y_pred_binary))


              precision    recall  f1-score   support

           0       1.00      0.19      0.32    723067
           1       0.04      1.00      0.08     26933

    accuracy                           0.22    750000
   macro avg       0.52      0.59      0.20    750000
weighted avg       0.97      0.22      0.31    750000

